# 4 · Conversation scoping (`conversation_id`)

Tie memories to a conversation so each conversation can keep its own context.
Pass `conversation_id` to any surface (`EngramMiddleware`, `create_memory_tools`,
`EngramStore`); it is attached as your project's scope property on writes and, with
`scope_recall_to_conversation=True`, added to the search filter.

> **Naming — the scope property is defined by *your* Engram project.** When you create a
> project you choose the scope property name; it is commonly `conversation_id` or `session_id`
> (the same concept LangGraph calls `thread_id`). `conversation_id` is *not* a reserved
> LangChain key — this package uses it only as a friendly parameter name for the *value*, and
> stores it under the property named by `conversation_property` (default `conversation_id`).
> **Set `conversation_property` to match your project's scope property** (e.g. `session_id`).
> Below we use a `SCOPE_PROPERTY` variable — change it to match your project.

> **Prerequisite — your Engram project must have the *conversation scope* available.**
> For `conversation_id` to actually isolate memories, your project's group/pipeline must
> define a topic that is **scoped by your scope property** (e.g. a `ConversationSummary`
> topic scoped by `conversation_id` / `session_id`). Configure this in the Engram console.
>
> Why it matters: in Engram only `user_id` and project are absolute boundaries. The scope
> property is a true isolation boundary **only** when the target topic is scoped by it. If
> the conversation scope is **not** available:
> - reconciliation runs at the **user** level, so contradictory facts from different
>   conversations merge into one **unscoped** memory (`properties=None`);
> - a property-filtered search returns that conversation's memories **plus** those
>   unscoped memories — so the merged fact shows up in *every* conversation.
>
> This package always sends `properties={<conversation_property>: ...}` on add and search,
> so once the scope is available and `conversation_property` matches, isolation just works.

## Setup — pick **one** option

**Option A — editable install (recommended).** Installs this package *and* its
dependencies from source; edits to `langchain_engram/` are picked up after a
kernel restart.

**Option B — import from local source (no install of `langchain-engram`).** Adds
the repo root to `sys.path` so `import langchain_engram` resolves straight to the
source tree. You still need the runtime dependencies available — the easiest is to
launch Jupyter from the repo's environment: `uv run --with jupyter jupyter lab`.
> For the agent cells you also need `langchain-anthropic` (Option A installs it; for Option B run `%pip install langchain-anthropic`).

In [ ]:
# Option A — editable install from source.
%pip install -q -e ".." langchain-anthropic

In [ ]:
# Option B — use langchain-engram straight from the local source tree (no install).
# Run this INSTEAD of Option A.
import sys
from pathlib import Path

repo_root = Path.cwd().parent  # notebooks/ -> repo root (holds langchain_engram/)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import langchain_engram

print('langchain_engram imported from:', langchain_engram.__file__)

In [ ]:
import getpass
import os

if not os.environ.get("ENGRAM_API_KEY"):
    os.environ["ENGRAM_API_KEY"] = getpass.getpass("ENGRAM_API_KEY: ")

# Optional: point at a non-default Engram endpoint.
# os.environ["ENGRAM_BASE_URL"] = "https://..."

In [ ]:
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')

In [ ]:
import time

from langchain_engram._client import build_client

client = build_client()  # reads ENGRAM_API_KEY

# If your Engram group has SCOPED TOPICS, every WRITE must include the required
# scope properties, or the API rejects it with, e.g.:
#   APIError: group 'default': insufficient scope: missing required scope
#   properties [custom_scope_1 some_other] to write memories
# Fill in the scope properties your project requires (leave {} if none):
SCOPE_PROPERTIES = {}  # e.g. {'custom_scope_1': 'demo', 'some_other': 'demo'}


def _merge_props(properties):
    merged = {**SCOPE_PROPERTIES, **(properties or {})}
    return merged or None


def seed_memory(text, user_id, *, properties=None, **kwargs):
    """Add a memory (with required scope properties) and wait for it to commit."""
    run = client.memories.add(
        text, user_id=user_id, properties=_merge_props(properties), **kwargs
    )
    status = client.runs.wait(run.run_id, timeout=60.0)
    print(f'run {run.run_id}: {status.status} '
          f'(+{len(status.memories_created)} / ~{len(status.memories_updated)})')
    return status


def wait_until_searchable(query, user_id, needle, timeout=60.0, *, properties=None, **kwargs):
    """Poll search until `needle` shows up (handles eventual consistency)."""
    props = _merge_props(properties)
    deadline = time.time() + timeout
    while time.time() < deadline:
        results = client.memories.search(
            query=query, user_id=user_id, properties=props, **kwargs
        )
        if any(needle.lower() in m.content.lower() for m in results):
            return results
        time.sleep(2.0)
    return client.memories.search(
        query=query, user_id=user_id, properties=props, **kwargs
    )

### What happens with / without the conversation scope (client-level)
Seed two *contradictory* seat preferences for the same user in different
conversations, then watch reconciliation and filtering.

In [ ]:
import uuid

# Set this to your project's scope property: 'conversation_id', 'session_id', ...
SCOPE_PROPERTY = 'conversation_id'

# fresh user per run so reconciliation isn't polluted by earlier runs
USER = f'conv-demo-{uuid.uuid4().hex[:8]}'

seed_memory('The user wants a window seat.', user_id=USER,
            properties={SCOPE_PROPERTY: 'trip-tokyo'})
seed_memory('The user wants an aisle seat.', user_id=USER,
            properties={SCOPE_PROPERTY: 'trip-lisbon'})

In [ ]:
print('All seat memories (cross-conversation):')
for m in client.memories.search(query='seat', user_id=USER):
    print('  -', m.content, '|', m.properties)

print('\nFiltered to the Tokyo conversation:')
for m in client.memories.search(query='seat', user_id=USER,
                                properties={SCOPE_PROPERTY: 'trip-tokyo'}):
    print('  -', m.content, '|', m.properties)

# WITHOUT the conversation scope: a reconciled, UNSCOPED memory (properties=None)
# appears in BOTH lists (the default topic is user-scoped). trip-lisbon is excluded
# from the Tokyo filter, but the unscoped/merged memory is not.
# WITH a topic scoped by SCOPE_PROPERTY: only that conversation's memories come back,
# and the merged/unscoped memory does not appear.

### Through `EngramMiddleware`
Bind a fixed `conversation_id`, point `conversation_property` at your project's scope
property, and turn on `scope_recall_to_conversation` so the recall search is filtered
to the conversation. (See step 5 to also pin recall to a topic like
`ConversationSummary`.)

In [ ]:
from langchain.agents import create_agent

from langchain_engram import EngramMiddleware

agent = create_agent(
    'anthropic:claude-sonnet-4-6',
    middleware=[EngramMiddleware(
        user_id=USER,
        conversation_id='trip-tokyo',
        conversation_property=SCOPE_PROPERTY,
        scope_recall_to_conversation=True,
    )],
)
out = agent.invoke(
    {'messages': [{'role': 'user', 'content': 'Which seat do I want for this trip?'}]}
)
print(out['messages'][-1].content)

### Resolve `conversation_id` at runtime
Read both `user_id` and `conversation_id` from the agent runtime context, so one
agent serves many users and conversations.

In [ ]:
from typing import TypedDict


class ConvCtx(TypedDict):
    user_id: str
    conversation_id: str


ctx_agent = create_agent(
    'anthropic:claude-sonnet-4-6',
    middleware=[EngramMiddleware(
        conversation_property=SCOPE_PROPERTY,
        scope_recall_to_conversation=True,
    )],
    context_schema=ConvCtx,
)
out = ctx_agent.invoke(
    {'messages': [{'role': 'user', 'content': 'Which seat do I want?'}]},
    context={'user_id': USER, 'conversation_id': 'trip-lisbon'},
)
print(out['messages'][-1].content)